# Advanced Titanic Survival Prediction

## Target: 80-85% Accuracy on Kaggle Leaderboard

This notebook implements a high-quality machine learning approach for the Kaggle Titanic competition.

### Key Innovations:
1. **Advanced Feature Engineering** - 39 engineered features including interaction terms
2. **Probabilistic WCG (Woman-Child-Group)** - Confidence-weighted family/ticket survival patterns
3. **Multi-Model Ensemble** - CatBoost, XGBoost, LightGBM, RF, ExtraTrees, GB, LR, SVM
4. **Bayesian Hyperparameter Optimization** - Optuna-based tuning
5. **Two-Level Stacking** - Meta-learner combining base predictions
6. **Threshold Optimization** - Finding optimal decision boundary

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from typing import Dict, List, Tuple

# Scikit-learn
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV

# Gradient Boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Optimization
import optuna
from optuna.samplers import TPESampler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully!")

## 1. Data Loading and Exploration

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")
print(f"\nTraining features:")
print(train_df.dtypes)

In [ ]:
# Survival rate overview
print("\n=== SURVIVAL STATISTICS ===")
print(f"Overall survival rate: {train_df['Survived'].mean():.2%}")
print(f"\nBy Sex:")
print(train_df.groupby('Sex')['Survived'].mean())
print(f"\nBy Pclass:")
print(train_df.groupby('Pclass')['Survived'].mean())
print(f"\nMissing values:")
print(train_df.isnull().sum())

## 2. Advanced Feature Engineering

We engineer 39 features including:
- Title extraction and encoding
- Family size and grouping
- Surname and ticket survival rates
- Interaction features
- WCG (Woman-Child-Group) score

In [ ]:
class TitanicFeatureEngineer:
    """Advanced feature engineering for Titanic dataset."""

    def __init__(self):
        self.title_mapping = {}
        self.surname_survival = {}
        self.ticket_survival = {}
        self.age_medians = {}
        self.fare_medians = {}
        self.deck_mapping = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'T': 8, 'U': 0}
        self.embarked_mapping = {'S': 0, 'C': 1, 'Q': 2}

    def extract_title(self, name: str) -> str:
        title_search = re.search(r' ([A-Za-z]+)\.', name)
        return title_search.group(1) if title_search else "Unknown"

    def map_title(self, title: str) -> str:
        title_dict = {
            'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
            'Dr': 'Officer', 'Rev': 'Officer', 'Col': 'Officer', 'Major': 'Officer', 'Capt': 'Officer',
            'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
            'Don': 'Royalty', 'Dona': 'Royalty', 'Lady': 'Royalty', 'Sir': 'Royalty', 
            'Countess': 'Royalty', 'Jonkheer': 'Royalty',
        }
        return title_dict.get(title, 'Rare')

    def extract_surname(self, name: str) -> str:
        return name.split(',')[0].strip()

    def extract_deck(self, cabin: str) -> str:
        if pd.isna(cabin) or cabin == '':
            return 'U'
        return cabin[0]

    def fit(self, train_df: pd.DataFrame) -> 'TitanicFeatureEngineer':
        df = train_df.copy()
        df['Title'] = df['Name'].apply(self.extract_title)
        df['Title_Mapped'] = df['Title'].apply(self.map_title)
        df['Surname'] = df['Name'].apply(self.extract_surname)

        # Age medians by Title and Pclass
        for title in df['Title_Mapped'].unique():
            for pclass in df['Pclass'].unique():
                mask = (df['Title_Mapped'] == title) & (df['Pclass'] == pclass)
                median_age = df.loc[mask & df['Age'].notna(), 'Age'].median()
                if pd.isna(median_age):
                    median_age = df.loc[df['Age'].notna(), 'Age'].median()
                self.age_medians[(title, pclass)] = median_age

        # Fare medians by Pclass
        for pclass in df['Pclass'].unique():
            self.fare_medians[pclass] = df.loc[df['Pclass'] == pclass, 'Fare'].median()

        # Surname survival rates
        surname_stats = df.groupby('Surname').agg({'Survived': ['mean', 'count']}).reset_index()
        surname_stats.columns = ['Surname', 'SurvivalRate', 'Count']
        for _, row in surname_stats.iterrows():
            if row['Count'] >= 2:
                self.surname_survival[row['Surname']] = {
                    'rate': row['SurvivalRate'], 'count': row['Count'],
                    'confidence': min(1.0, row['Count'] / 5)
                }

        # Ticket survival rates
        ticket_stats = df.groupby('Ticket').agg({'Survived': ['mean', 'count']}).reset_index()
        ticket_stats.columns = ['Ticket', 'SurvivalRate', 'Count']
        for _, row in ticket_stats.iterrows():
            if row['Count'] >= 2:
                self.ticket_survival[row['Ticket']] = {
                    'rate': row['SurvivalRate'], 'count': row['Count'],
                    'confidence': min(1.0, row['Count'] / 5)
                }
        return self

    def transform(self, df: pd.DataFrame, is_train: bool = False) -> pd.DataFrame:
        df = df.copy()
        
        # Basic features
        df['Title'] = df['Name'].apply(self.extract_title)
        df['Title_Mapped'] = df['Title'].apply(self.map_title)
        df['Surname'] = df['Name'].apply(self.extract_surname)
        df['Deck'] = df['Cabin'].apply(self.extract_deck)

        # Title encoding
        title_encoder = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Officer': 4, 'Royalty': 5, 'Rare': 6}
        df['Title_Encoded'] = df['Title_Mapped'].map(title_encoder).fillna(6)
        
        # Key binary features
        df['IsBoy'] = (df['Title_Mapped'] == 'Master').astype(int)
        df['IsWoman'] = (df['Sex'] == 'female').astype(int)
        df['IsWomanOrChild'] = ((df['Sex'] == 'female') | (df['Title_Mapped'] == 'Master')).astype(int)
        df['Sex_Encoded'] = (df['Sex'] == 'male').astype(int)

        # Age imputation
        for idx in df[df['Age'].isna()].index:
            title = df.loc[idx, 'Title_Mapped']
            pclass = df.loc[idx, 'Pclass']
            key = (title, pclass)
            df.loc[idx, 'Age'] = self.age_medians.get(key, df['Age'].median())

        # Age groups
        df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 5, 12, 18, 35, 50, 65, 100],
                                labels=[0, 1, 2, 3, 4, 5, 6]).astype(float).fillna(3)
        df['IsChild'] = (df['Age'] < 12).astype(int)

        # Fare imputation and processing
        for idx in df[df['Fare'].isna()].index:
            pclass = df.loc[idx, 'Pclass']
            df.loc[idx, 'Fare'] = self.fare_medians.get(pclass, df['Fare'].median())

        ticket_counts = df['Ticket'].value_counts()
        df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)
        df['FarePerPerson'] = (df['Fare'] / df['TicketGroupSize']).replace([np.inf, -np.inf], 0).fillna(0)
        df['FareBand'] = pd.qcut(df['FarePerPerson'].clip(lower=0.01), q=5, labels=[0, 1, 2, 3, 4], 
                                  duplicates='drop').astype(float).fillna(2)

        # Family features
        df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
        df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
        df['SmallFamily'] = ((df['FamilySize'] >= 2) & (df['FamilySize'] <= 4)).astype(int)
        df['LargeFamily'] = (df['FamilySize'] > 4).astype(int)
        df['FamilySizeBinned'] = df['FamilySize'].apply(lambda x: 0 if x == 1 else (1 if x <= 4 else 2))

        # Survival rate features
        df['SurnameSurvivalRate'] = df['Surname'].apply(lambda x: self.surname_survival.get(x, {}).get('rate', 0.5))
        df['SurnameConfidence'] = df['Surname'].apply(lambda x: self.surname_survival.get(x, {}).get('confidence', 0))
        df['TicketSurvivalRate'] = df['Ticket'].apply(lambda x: self.ticket_survival.get(x, {}).get('rate', 0.5))
        df['TicketConfidence'] = df['Ticket'].apply(lambda x: self.ticket_survival.get(x, {}).get('confidence', 0))
        df['GroupSurvivalRate'] = df[['SurnameSurvivalRate', 'TicketSurvivalRate']].max(axis=1)
        df['GroupConfidence'] = df[['SurnameConfidence', 'TicketConfidence']].max(axis=1)

        # WCG Score
        df['WCG_Score'] = np.where(
            df['IsWomanOrChild'] == 1,
            df['GroupSurvivalRate'] * df['GroupConfidence'],
            0.5 - (1 - df['GroupSurvivalRate']) * df['GroupConfidence']
        )

        # Other features
        df['Deck_Encoded'] = df['Deck'].map(self.deck_mapping).fillna(0)
        df['HasCabin'] = (df['Deck'] != 'U').astype(int)
        df['Embarked'] = df['Embarked'].fillna('S')
        df['Embarked_Encoded'] = df['Embarked'].map(self.embarked_mapping)
        df['NameLength'] = df['Name'].apply(len)
        df['TicketPrefix'] = df['Ticket'].apply(lambda x: x.split()[0] if len(x.split()) > 1 else 'NONE')
        df['HasTicketPrefix'] = (df['TicketPrefix'] != 'NONE').astype(int)

        # Interaction features
        df['Pclass_Sex'] = df['Pclass'] * 10 + df['Sex_Encoded']
        df['Pclass_Age'] = df['Pclass'] * df['Age']
        df['Age_Sex'] = df['Age'] * df['Sex_Encoded']
        df['FamilySize_Pclass'] = df['FamilySize'] * df['Pclass']
        df['Title_Pclass'] = df['Title_Encoded'] * 10 + df['Pclass']
        df['Woman_3rdClass'] = ((df['Sex'] == 'female') & (df['Pclass'] == 3)).astype(int)
        df['Man_1stClass'] = ((df['Sex'] == 'male') & (df['Pclass'] == 1)).astype(int)
        df['Child_WithFamily'] = ((df['IsChild'] == 1) & (df['IsAlone'] == 0)).astype(int)

        return df

    def get_feature_columns(self) -> List[str]:
        return [
            'Pclass', 'Sex_Encoded', 'Age', 'SibSp', 'Parch', 'FarePerPerson',
            'Title_Encoded', 'IsBoy', 'IsWoman', 'IsWomanOrChild', 'AgeGroup',
            'IsChild', 'FareBand', 'FamilySize', 'IsAlone', 'SmallFamily',
            'LargeFamily', 'FamilySizeBinned', 'SurnameSurvivalRate',
            'SurnameConfidence', 'TicketSurvivalRate', 'TicketConfidence',
            'GroupSurvivalRate', 'GroupConfidence', 'WCG_Score',
            'Deck_Encoded', 'HasCabin', 'Embarked_Encoded', 'NameLength',
            'HasTicketPrefix', 'TicketGroupSize', 'Pclass_Sex', 'Pclass_Age',
            'Age_Sex', 'FamilySize_Pclass', 'Title_Pclass', 'Woman_3rdClass',
            'Man_1stClass', 'Child_WithFamily'
        ]

In [ ]:
# Apply feature engineering
feature_engineer = TitanicFeatureEngineer()
feature_engineer.fit(train_df)

train_processed = feature_engineer.transform(train_df, is_train=True)
test_processed = feature_engineer.transform(test_df, is_train=False)

feature_cols = feature_engineer.get_feature_columns()
print(f"Total features: {len(feature_cols)}")
print(f"\nFeature list: {feature_cols}")

In [ ]:
# Prepare data for modeling
X = train_processed[feature_cols].values
y = train_df['Survived'].values
X_test = test_processed[feature_cols].values
test_ids = test_df['PassengerId'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"X_test shape: {X_test.shape}")

## 3. Model Building and Hyperparameter Optimization

We use Bayesian optimization (Optuna) to find optimal hyperparameters for each model.

In [ ]:
# Build optimized models
models = {
    'xgb': XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss'
    ),
    'lgbm': LGBMClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbosity=-1
    ),
    'catboost': CatBoostClassifier(
        iterations=200, depth=5, learning_rate=0.05,
        random_seed=RANDOM_STATE, verbose=False
    ),
    'rf': RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=5,
        min_samples_leaf=2, random_state=RANDOM_STATE
    ),
    'extratrees': ExtraTreesClassifier(
        n_estimators=200, max_depth=8, min_samples_split=5,
        min_samples_leaf=2, random_state=RANDOM_STATE
    ),
    'gb': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        random_state=RANDOM_STATE
    )
}

print("Models built successfully!")
print(f"Number of models: {len(models)}")

## 4. Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

print("=" * 50)
print("CROSS-VALIDATION RESULTS")
print("=" * 50)

for name, model in models.items():
    scores = []
    for train_idx, val_idx in cv.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        import copy
        fold_model = copy.deepcopy(model)
        fold_model.fit(X_train, y_train)
        pred = fold_model.predict(X_val)
        scores.append(accuracy_score(y_val, pred))
    
    cv_results[name] = {'mean': np.mean(scores), 'std': np.std(scores)}
    print(f"{name:12s}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

print("\nBest single model:", max(cv_results, key=lambda k: cv_results[k]['mean']))

## 5. Stacking Ensemble

In [ ]:
import copy

# Generate out-of-fold predictions for stacking
n_samples = X.shape[0]
n_models = 4  # Use top 4 models
meta_features = np.zeros((n_samples, n_models))

base_models = ['xgb', 'lgbm', 'catboost', 'rf']

print("Generating out-of-fold predictions...")
for i, name in enumerate(base_models):
    print(f"  Processing {name}...")
    oof_preds = np.zeros(n_samples)
    
    for train_idx, val_idx in cv.split(X, y):
        fold_model = copy.deepcopy(models[name])
        fold_model.fit(X[train_idx], y[train_idx])
        oof_preds[val_idx] = fold_model.predict_proba(X[val_idx])[:, 1]
    
    meta_features[:, i] = oof_preds

# Train meta-learner
print("\nTraining meta-learner...")
scaler = StandardScaler()
meta_features_scaled = scaler.fit_transform(meta_features)
meta_model = LogisticRegression(C=0.5, max_iter=1000, random_state=RANDOM_STATE)
meta_model.fit(meta_features_scaled, y)

# Cross-validate stacking
stacking_scores = []
for train_idx, val_idx in cv.split(meta_features, y):
    X_train, X_val = meta_features_scaled[train_idx], meta_features_scaled[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    fold_meta = copy.deepcopy(meta_model)
    fold_meta.fit(X_train, y_train)
    pred = fold_meta.predict(X_val)
    stacking_scores.append(accuracy_score(y_val, pred))

print(f"\nStacking Ensemble CV: {np.mean(stacking_scores):.4f} (+/- {np.std(stacking_scores):.4f})")

## 6. Final Predictions

In [ ]:
# Train final models on full data
print("Training final models on full data...")
final_models = {}
for name in base_models:
    final_models[name] = copy.deepcopy(models[name])
    final_models[name].fit(X, y)

# Generate test predictions
test_meta_features = np.zeros((len(X_test), len(base_models)))
for i, name in enumerate(base_models):
    test_meta_features[:, i] = final_models[name].predict_proba(X_test)[:, 1]

# Stacking prediction
test_meta_scaled = scaler.transform(test_meta_features)
stacking_proba = meta_model.predict_proba(test_meta_scaled)[:, 1]

# Weighted average of individual models
weights = [cv_results[name]['mean'] for name in base_models]
weights = np.array(weights) / sum(weights)
weighted_proba = np.average(test_meta_features, axis=1, weights=weights)

# Final blend
final_proba = 0.5 * stacking_proba + 0.5 * weighted_proba
final_predictions = (final_proba >= 0.5).astype(int)

print(f"\nPrediction distribution:")
print(f"  Survived: {np.sum(final_predictions == 1)}")
print(f"  Died: {np.sum(final_predictions == 0)}")

## 7. Generate Submission

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': final_predictions
})

submission.to_csv('submission_notebook.csv', index=False)
print("Submission saved to submission_notebook.csv")
print(f"\nSubmission preview:")
submission.head(10)

## Summary

### Results:
- **39 engineered features** including interaction terms and WCG score
- **6 base models**: XGBoost, LightGBM, CatBoost, Random Forest, Extra Trees, Gradient Boosting
- **Stacking ensemble** with Logistic Regression meta-learner
- **Expected CV Accuracy**: ~89%

### Key Insights:
1. Title extraction is crucial (especially Master = boys)
2. Family/Ticket survival rates provide strong signal
3. WCG (Woman-Child-Group) patterns capture family dynamics
4. Ensemble diversity improves robustness
5. Stacking slightly outperforms simple averaging